In [1]:
# CODE BLOCK 1: Setup and Tool Definitions

import pandas as pd
import numpy as np
from typing import Dict, Any, List

# --- SIMULATED DATA AND FORECASTING MODEL ---
def get_historical_sales(sku: str, date_range: str = '3M') -> pd.DataFrame:
    """Retrieves structured historical sales data for a given SKU."""
    print(f"-> TOOL CALL: Retrieving sales for {sku} over the last {date_range}.")
    
    # Simulate historical data
    dates = pd.date_range(end='2025-10-31', periods=90, freq='D')
    sales = np.random.randint(100, 300, size=len(dates))
    
    # Create a subtle trend based on the product (e.g., if SKU contains 'A')
    if 'A' in sku:
        sales = sales + np.linspace(0, 50, len(dates))

    df = pd.DataFrame({'Date': dates, 'Sales': sales.round().astype(int)})
    return df

def run_forecasting_model(sales_data: pd.DataFrame, features: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    Executes a quantitative time-series model (e.g., ARIMA or Prophet).
    In a real system, this would call a trained statistical library.
    """
    print("-> TOOL CALL: Executing statistical forecasting model...")
    
    # Get last known sales average
    last_average = sales_data['Sales'].tail(10).mean()
    
    # Simple linear forecast (PLACEHOLDER for a complex model)
    forecast_units = last_average * 30 # Forecast for next 30 days
    
    # Apply LLM-generated feature adjustments (e.g., a "Hype Score")
    adjustment_factor = 1.0
    for feature in features:
        if feature.get('name') == 'Social_Media_Hype':
            adjustment_factor += feature['value'] * 0.1 # Example: Hype adds 10%
        if feature.get('name') == 'Competitor_Launch':
            adjustment_factor -= 0.15 # Example: Launch subtracts 15%
            
    final_forecast = int(forecast_units * adjustment_factor)
    
    return {
        "forecast_units": final_forecast,
        "base_projection": int(forecast_units),
        "adjustment_factor": adjustment_factor
    }

def adjust_inventory_api(sku: str, quantity: int) -> str:
    """Simulates an API call to update the inventory system."""
    print(f"-> TOOL CALL: Updating ERP/Inventory system...")
    return f"Inventory system updated: Safety stock for {sku} set to {quantity} units."

In [6]:
# CODE BLOCK 2: LLM Agent Core and Orchestration

# --- SIMULATED LLM REASONING (The 'Brain' of the Agent) ---
def llm_reasoning_and_orchestration(user_query: str) -> str:
    """
    Simulates the LLM's role in interpreting the query, generating contextual 
    features, calling tools, and synthesizing the final response.
    """
    print(f"--- LLM Agent received query: '{user_query}' ---")
    
    # 1. Query Interpretation and Intent (Simulated by simple checks)
    if 'size 8 red t-shirt' in user_query.lower():
        sku = "TSHIRT-RED-A"
        forecast_period = "30 days"
    else:
        return "ERROR: Could not identify target product SKU."

    # 2. Contextual Feature Generation (RAG/External Data Integration)
    # In a real system, the LLM would query a vector store/API based on the SKU.
    print(f"--- LLM REASONING: Checking external context for {sku} ---")
    
    # Simulated Unstructured Context:
    recent_news = "Competitor B just announced a major winter apparel line launch."
    social_sentiment = "Sentiment analysis shows a strong spike in positive engagement for SKU-A."
    
    # LLM processes text to generate structured features for the quantitative model
    contextual_features = [
        {"name": "Social_Media_Hype", "value": 0.5, "source": "sentiment_analysis"}, # 50% positive spike
        {"name": "Competitor_Launch", "value": 1, "source": "news_feed"}, # Competitor launch is happening
    ]
    
    print(f"--- LLM generated features: {contextual_features} ---")

    # 3. Tool Execution: Data Retrieval
    historical_data = get_historical_sales(sku)

    print("historical_data: ", historical_data.head())
    
    # 4. Tool Execution: Forecasting
    forecast_result = run_forecasting_model(historical_data, contextual_features)

    # 5. Synthesis and Action
    
    final_forecast_units = forecast_result['forecast_units']
    
    # LLM recommends an action based on the forecast
    recommended_safety_stock = int(final_forecast_units * 1.5) # Example safety margin
    
    # 6. Tool Execution: API Integration (Optional Action Step)
    inventory_status = adjust_inventory_api(sku, recommended_safety_stock)

    # 7. Final Response Generation (Explaining the result)
    explanation = f"""
    ✅ **Forecast Complete for {sku}**

    * **Base Statistical Projection:** {forecast_result['base_projection']} units.
    * **Contextual Adjustment:** Adjusted by a factor of {forecast_result['adjustment_factor']:.2f} due to external factors.
    * **Final Forecast (30 days):** **{final_forecast_units} units.**
    
    ---
    
    **Reasoning:** The initial projection was adjusted because the system detected:
    1.  A **strong positive spike** in social media hype (factor +5%).
    2.  A **major competitor launch** announcement (factor -15%).
    
    The net effect was a **reduction** in the final forecast.
    
    **Action:** {inventory_status}
    """
    
    return explanation.strip()

In [7]:
# CODE BLOCK 3: Execution
if __name__ == "__main__":
    # --- RUN THE AGENT ---
    user_request = "Forecast demand for size 8 red t-shirts next month and update safety stock."
    
    final_report = llm_reasoning_and_orchestration(user_request)
    
    print("\n" + "="*50)
    print("FINAL AGENT REPORT")
    print("="*50)
    print(final_report)
    print("="*50)

--- LLM Agent received query: 'Forecast demand for size 8 red t-shirts next month and update safety stock.' ---
--- LLM REASONING: Checking external context for TSHIRT-RED-A ---
--- LLM generated features: [{'name': 'Social_Media_Hype', 'value': 0.5, 'source': 'sentiment_analysis'}, {'name': 'Competitor_Launch', 'value': 1, 'source': 'news_feed'}] ---
-> TOOL CALL: Retrieving sales for TSHIRT-RED-A over the last 3M.
historical_data:          Date  Sales
0 2025-08-03    264
1 2025-08-04    273
2 2025-08-05    151
3 2025-08-06    185
4 2025-08-07    186
-> TOOL CALL: Executing statistical forecasting model...
-> TOOL CALL: Updating ERP/Inventory system...

FINAL AGENT REPORT
✅ **Forecast Complete for TSHIRT-RED-A**

    * **Base Statistical Projection:** 6957 units.
    * **Contextual Adjustment:** Adjusted by a factor of 0.90 due to external factors.
    * **Final Forecast (30 days):** **6261 units.**

    ---

    **Reasoning:** The initial projection was adjusted because the system de